##### full code - All Pages Active Contracts Download

In [5]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from datetime import datetime

# Setup download directory
download_dir = os.path.join(os.getcwd(), "fdot_downloads_" + datetime.now().strftime("%Y-%b-%d"))
os.makedirs(download_dir, exist_ok=True)

# Setup Chrome WebDriver
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_setting_values.automatic_downloads": 1
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

processed_contracts = set()

def wait_for_download(before_files, timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        current_files = set(os.listdir(download_dir))
        new_files = current_files - before_files
        for f in new_files:
            if f.endswith(".xlsx") or f.endswith(".crdownload"):
                return f
        time.sleep(1)
    return None


try:
    driver.get("https://scoc.fdot.gov/#/active/1")
    wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))

    page_count = 1

    while True:
        print(f"\n--- Processing Page {page_count} ---\n")

        wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))
        rows = driver.find_elements(By.CSS_SELECTOR, 'table[aria-label="Contracts"] tbody tr')

        row_index = 0
        while row_index < len(rows):
            row = rows[row_index]
            row_index += 1

            cells = row.find_elements(By.TAG_NAME, "td")
            if not cells:
                continue

            contract_id = cells[2].text.strip()
            is_active = cells[0].text.strip().lower() == "yes"

            if contract_id in processed_contracts or not is_active:
                continue

            processed_contracts.add(contract_id)
            print(f"Processing Contract ID: {contract_id}")

            try:
                link = cells[2].find_element(By.XPATH, './/a[@aria-label="View Contract Details"]')
                driver.execute_script("arguments[0].click();", link)
                time.sleep(1.5)

                wait.until(EC.presence_of_element_located((
                    By.XPATH,
                    '//div[contains(@class, "contract-detail")] | //a[contains(text(), "Back to Active Contract List")]'
                )))

                try:
                    report_button_xpath = f'//*[contains(@title, "Get estimate detail report for {contract_id}")]'
                    report_buttons = driver.find_elements(By.XPATH, report_button_xpath)

                    if not report_buttons:
                        print(f"[{contract_id}] No estimate detail report available. Skipping.")
                    else:
                        report_button = report_buttons[0]
                        driver.execute_script("arguments[0].click();", report_button)
                        time.sleep(1)

                        wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "modal-content")]')))
                        driver.save_screenshot(f"modal_{contract_id}.png")

                        format_dropdown = wait.until(EC.element_to_be_clickable((By.XPATH, '//select')))
                        excel_option = None
                        for option in format_dropdown.find_elements(By.TAG_NAME, 'option'):
                            if "Excel" in option.text:
                                excel_option = option
                                break

                        if excel_option:
                            driver.execute_script("""
                                arguments[0].selected = true;
                                arguments[0].dispatchEvent(new Event('change', { bubbles: true }));
                            """, excel_option)
                            print(f"[{contract_id}] Excel format selected.")
                            time.sleep(1)
                        else:
                            print(f"[{contract_id}] Excel option not found.")
                            continue

                        try:
                            wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, 'div.overlay')))
                        except TimeoutException:
                            print(f"[{contract_id}] Overlay did not disappear in time.")

                        before_files = set(os.listdir(download_dir))
                        run_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[contains(@title, "Run Report - Get Estimate Detail")]')))
                        driver.execute_script("arguments[0].click();", run_button)
                        print(f"[{contract_id}] Triggered Excel download.")

                        downloaded_file = wait_for_download(before_files, timeout=30)
                        if downloaded_file:
                            print(f"[{contract_id}] Download started")
                        else:
                            print(f"[{contract_id}] Download did not start within timeout.")

                        time.sleep(10)

                        try:
                            cancel_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@title="Cancel" and contains(@class, "close")]')))
                            driver.execute_script("arguments[0].click();", cancel_button)
                            time.sleep(1)
                        except TimeoutException:
                            print(f"[{contract_id}] Cancel button not found.")

                except Exception as e:
                    print(f"[{contract_id}] Failed to download report: {e}")

                back_button = driver.find_element(By.XPATH, '//a[contains(text(), "Back to Active Contract List")]')
                driver.execute_script("arguments[0].click();", back_button)
                wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))
                time.sleep(2)
                rows = driver.find_elements(By.CSS_SELECTOR, 'table[aria-label="Contracts"] tbody tr')

            except Exception as e:
                print(f"[{contract_id}] Error during processing: {e}")
                driver.save_screenshot(f"error_{contract_id}.png")
                driver.get("https://scoc.fdot.gov/#/active/1")
                time.sleep(2)
                break

        # Try to go to the next page
        try:
            next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@title="next page"]')))
            if "disabled" in next_button.get_attribute("class"):
                print("Next button is disabled. Reached last page.")
                break
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(2)
            page_count += 1
        except TimeoutException:
            print("Next Page button not found or not clickable. Ending pagination.")
            break

finally:
    driver.quit()

KeyboardInterrupt: 

##### Get Latest List of all Projects from FDOT website - run this code once after each cut off

In [35]:
import os
import time
import shutil
import glob
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

# Get Contracts Data till last month
df_last = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Combined_Till_Last_Month.xlsx")
print(f"Loaded {df_last.shape[0]} contracts from last month.")
df_latest = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Latest.xlsx")
print(f"Loaded {df_latest.shape[0]} contracts from latest data.")
merged_df = pd.concat([df_last, df_latest], ignore_index=True)
print(f"Merged DataFrame shape: {merged_df.shape}")
merged_df = merged_df.drop_duplicates()
print(f"After dropping duplicates, DataFrame shape: {merged_df.shape}")
# Save the merged DataFrame to a new Excel file
output_file = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Combined_Till_Last_Month.xlsx"
merged_df.to_excel(output_file, index=False)

Loaded 2873 contracts from last month.
Loaded 1489 contracts from latest data.
Merged DataFrame shape: (4362, 30)
After dropping duplicates, DataFrame shape: (3542, 30)


In [36]:
# Set download directory
root_dir=r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data"
# root_dir = input("Enter the folder path where All Contracts should be downloaded:  ")
os.makedirs(root_dir, exist_ok=True)

# Set up Chrome options
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": root_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True}
options.add_experimental_option("prefs", prefs)

# Start browser
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

# Open the URL
url = 'https://scoc.fdot.gov/#/active/1'
driver.get(url)

# Wait for overlay to disappear
try:
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, "div.overlay")))
except:
    pass

# Wait for the Excel export button
excel_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//span[contains(@class, "fa-file-excel")]')))

# Scroll into view and click
driver.execute_script("arguments[0].scrollIntoView(true);", excel_button)
time.sleep(1)
try:
    excel_button.click()
except:
    driver.execute_script("arguments[0].click();", excel_button)

# Wait for download to complete
time.sleep(30)  # Adjust if needed

# Get list of all Excel files in the download directory
excel_files = glob.glob(os.path.join(root_dir, "*.xlsx"))
#r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\'
if not excel_files:
    print("No Excel files found.")
    driver.quit()
    exit()

# Sort files by modification time (latest first)
latest_file = max(excel_files, key=os.path.getmtime)

today = datetime.today().strftime('%Y-%b-%d')
# Rename the latest file
desired_filename = "FDOT_All_Contracts_Latest.xlsx"
desired_filepath = os.path.join(root_dir, desired_filename)
print(desired_filepath)


if os.path.exists(desired_filepath):
    os.remove(desired_filepath)

os.rename(latest_file, desired_filepath)
# Clean up
driver.quit()
# print(f"Filtered data saved to: {output_file}")

df = pd.read_excel(desired_filepath)
# df = df[df["Active"] == True]
df.to_excel(desired_filepath, index=False)

C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Latest.xlsx


#### Compare latest and last months'contracts data for Payment status

In [1]:
import os
import time
import shutil
import glob
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

# Set download directory
root_dir = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data"
os.makedirs(root_dir, exist_ok=True)

contracts_df_latest=pd.read_excel(root_dir + r"\FDOT_All_Contracts_Latest.xlsx")
contracts_df_latest=contracts_df_latest[['Contract ID', 'Status']].copy().drop_duplicates()
contracts_df_latest = contracts_df_latest[contracts_df_latest['Status'] == 'FINAL PAYMENT MADE']
contracts_df_latest = contracts_df_latest.rename(columns={'Contract ID': 'Contract'})

contracts_df_last_month=pd.read_excel(root_dir + r"\FDOT_All_Contracts_Combined_Till_Last_Month.xlsx")
contracts_df_last_month=contracts_df_last_month[['Contract ID', 'Status']].copy().drop_duplicates()
contracts_df_last_month = contracts_df_last_month[contracts_df_last_month['Status'] == 'FINAL PAYMENT MADE'].drop(columns=['Status'])
contracts_df_last_month = contracts_df_last_month.rename(columns={'Contract ID': 'Contract'})

# Merge contracts_df with df to get the status of each contract
common_contracts = pd.merge(contracts_df_latest, contracts_df_last_month, on='Contract', how='inner')
common_contracts = common_contracts[['Contract','Status']].copy().drop_duplicates()

#### code - Downloads based upon Master Job List, Status not equal to FINAL PAYMENT MADE, combine total output and filter on pay item list and last cut off date

In [144]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from selenium.common.exceptions import NoSuchElementException
from datetime import datetime


# Setup download directory
# root_dir = input("Enter the folder path where 'fdot_downloads' should be created: ")
root_dir = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data"
download_dir = os.path.join(root_dir, "fdot_downloads_" + datetime.now().strftime("%Y-%b-%d"))
os.makedirs(download_dir, exist_ok=True)
print(f"Download directory set to: {download_dir}")

# Read Master_Job_List
Master_Job_List = pd.read_excel(root_dir + r"\\Acme DOT Jobs (from Sage 100).xlsx")
Master_Job_List.columns = Master_Job_List.iloc[0]  # Set the first row as header
Master_Job_List = Master_Job_List.iloc[1:]
Master_Job_List = Master_Job_List[Master_Job_List['JobType'] == 'DOT'].copy()

# Setup Chrome WebDriver
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_setting_values.automatic_downloads": 1
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

processed_contracts = set()

try:
    driver.get("https://scoc.fdot.gov/#/active/1")
    wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))

    page_count = 1

    while True:
        # print(f"\n--- Processing Page {page_count} ---\n")

        wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))
        rows = driver.find_elements(By.CSS_SELECTOR, 'table[aria-label="Contracts"] tbody tr')

        row_index = 0
        while row_index < len(rows):
            row = rows[row_index]
            row_index += 1

            cells = row.find_elements(By.TAG_NAME, "td")
            if not cells or all(cell.text.strip() == "" for cell in cells):
                continue  # Skip blank rows silently

            contract_id = cells[2].text.strip()
            is_active = cells[0].text.strip().lower() == "yes"

            # ✅ Only process if in master list, not in common_contracts and not already processed
            if contract_id not in Master_Job_List["JobNo"].values:
                print(f"\nSkipping Contract ID: {contract_id} — not found in master job list.")
                continue
            if contract_id in common_contracts["Contract"].values:
                print(f"\nSkipping Contract ID: {contract_id} — has 'FINAL PAYMENT MADE' Status.")
                continue

            if contract_id in processed_contracts:
                print(f"\nSkipping Contract ID: {contract_id} — already processed.")
                continue

            processed_contracts.add(contract_id)
            print(f"\nProcessing Contract ID: {contract_id}")

            try:
                link = cells[2].find_element(By.XPATH, './/a[@aria-label="View Contract Details"]')
                driver.execute_script("arguments[0].click();", link)
                time.sleep(1.5)

                wait.until(EC.presence_of_element_located((
                    By.XPATH,
                    '//div[contains(@class, "contract-detail")] | //a[contains(text(), "Back to Active Contract List")]')))

                try:
                    report_button_xpath = f'//*[contains(@title, "Get estimate detail report for {contract_id}")]'
                    report_buttons = driver.find_elements(By.XPATH, report_button_xpath)

                    if not report_buttons:
                        print(f"[{contract_id}] No estimate detail report available. Skipping.")
                    else:
                        report_button = report_buttons[0]
                        driver.execute_script("arguments[0].click();", report_button)
                        time.sleep(1)

                        wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "modal-content")]')))
                        # driver.save_screenshot(f"modal_{contract_id}.png")

                        format_dropdown = wait.until(EC.element_to_be_clickable((By.XPATH, '//select')))
                        excel_option = None
                        for option in format_dropdown.find_elements(By.TAG_NAME, 'option'):
                            if "Excel" in option.text:
                                excel_option = option
                                break

                        if excel_option:
                            driver.execute_script("""
                                arguments[0].selected = true;
                                arguments[0].dispatchEvent(new Event('change', { bubbles: true }));
                            """, excel_option)
                            print(f"[{contract_id}] Excel format selected.")
                            time.sleep(1)
                        else:
                            print(f"[{contract_id}] Excel option not found.")
                            continue

                        try:
                            wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, 'div.overlay')))
                        except TimeoutException:
                            print(f"[{contract_id}] Overlay did not disappear in time.")

                        before_files = set(os.listdir(download_dir))
                        run_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[contains(@title, "Run Report - Get Estimate Detail")]')))
                        driver.execute_script("arguments[0].click();", run_button)
                        print(f"[{contract_id}] Triggered Excel download.")

                        time.sleep(30)

                        # Click on cancel button after triggering the download
                        # cancel_button = driver.find_element(By.XPATH, '//button[@title="Cancel" and contains(@class, "close")]')
                        # driver.execute_script("arguments[0].click();", cancel_button)


                        try:
                            cancel_button = WebDriverWait(driver, 5).until(
                                EC.presence_of_element_located((By.XPATH, '//button[@title="Cancel" and contains(@class, "close")]')))
                            driver.execute_script("arguments[0].click();", cancel_button)
                            print(f"[{contract_id}] Cancel button clicked.")
                            time.sleep(10)
                        except (TimeoutException, NoSuchElementException) as e:
                            pass

                except Exception as e:
                    print(f"[{contract_id}] Failed to download report: {e}")

                back_button = driver.find_element(By.XPATH, '//a[contains(text(), "Back to Active Contract List")]')
                driver.execute_script("arguments[0].click();", back_button)
                wait.until(EC.presence_of_element_located((By.XPATH, '//table[@aria-label="Contracts"]')))
                time.sleep(2)
                rows = driver.find_elements(By.CSS_SELECTOR, 'table[aria-label="Contracts"] tbody tr')

            except Exception as e:
                print(f"[{contract_id}] Error during processing: {e}")
                driver.save_screenshot(f"error_{contract_id}.png")
                driver.get("https://scoc.fdot.gov/#/active/1")
                time.sleep(2)
                break

        # Try to go to the next page
        try:
            next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@title="next page"]')))
            if "disabled" in next_button.get_attribute("class"):
                print("Next button is disabled. Reached last page.")
                break
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(2)
            page_count += 1
        except TimeoutException:
            print("Next Page button not found or not clickable. Ending pagination. All contracts processed.")
            break

finally:
    driver.quit()

Download directory set to: C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\fdot_downloads_2025-Oct-29

Skipping Contract ID: E1R18 — not found in master job list.

Skipping Contract ID: E1R76 — not found in master job list.

Processing Contract ID: E1R87
[E1R87] Excel format selected.
[E1R87] Triggered Excel download.

Processing Contract ID: E1S28
[E1S28] Excel format selected.
[E1S28] Triggered Excel download.

Skipping Contract ID: E1S92 — has 'FINAL PAYMENT MADE' Status.

Processing Contract ID: E1T28
[E1T28] Excel format selected.
[E1T28] Triggered Excel download.

Skipping Contract ID: E1U11 — not found in master job list.

Skipping Contract ID: E1U15 — has 'FINAL PAYMENT MADE' Status.

Skipping Contract ID: E1U16 — has 'FINAL PAYMENT MADE' Status.

Skipping Contract ID: E1U17 — not found in master job list.

Skipping Contract ID: E1U22 — not found in master job list.

Skipping Contract ID: E1U56 — not found in master job list.

Skippi

#### Combine all fdot downloaded contracts into 1

In [145]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
import re

# base_path = r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation'
base_path=root_dir
folder_paths = [os.path.join(base_path, name) 
                for name in os.listdir(base_path) 
                if os.path.isdir(os.path.join(base_path, name)) and "fdot_downloads" in name]
# Create a DataFrame from folder_paths
df_folders = pd.DataFrame({'folder_path': folder_paths})
# Extract the folder name (after last '\')
df_folders['folder_name'] = df_folders['folder_path'].apply(lambda x: os.path.basename(x))
# Extract the date using regex
df_folders['date_str'] = df_folders['folder_name'].str.extract(r'(\d{4}-[A-Za-z]{3}-\d{2})')
# Convert to datetime for comparison
df_folders['date'] = pd.to_datetime(df_folders['date_str'], format='%Y-%b-%d', errors='coerce')
# Find the row with the latest date
latest_row = df_folders.loc[df_folders['date'].idxmax()]
latest_folder_path = latest_row['folder_path']
print('latest folder path: ', latest_folder_path)

all_dataframes = []

for filename in os.listdir(latest_folder_path):
    if filename.endswith('.XLSX') or filename.endswith('.xlsx'):
        file_path = os.path.join(latest_folder_path, filename)
        try:
            df = pd.read_excel(file_path, engine='openpyxl')
            if not df.empty:
                all_dataframes.append(df)
            else:
                print(f"Skipped empty file: {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

if all_dataframes:
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    combined_df.columns = [col.replace('\n', '').strip() for col in combined_df.columns]
    combined_df=combined_df[['Contract','EstimateNumber', 'EstimateEnd Date','Project', 'Unit', 'Item Code', 'Previous', 
                             'This Est.', 'To-Date','Item Description']].copy().drop_duplicates()
    combined_df['Item Code'] = combined_df['Item Code'].str.strip().str.replace(r'\s+', ' ', regex=True)
    combined_df['Date_Downloaded'] = latest_row['date_str']
    
    print("All FDOT Downloads of Excel files have been combined into one file.")
else:
    print("No valid Excel files found to combine.")

combined_df.shape

latest folder path:  C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\fdot_downloads_2025-Oct-29
All FDOT Downloads of Excel files have been combined into one file.


(863144, 11)

In [146]:
combined_df['Contract'].nunique()

505

In [ ]:
trial = combined_df[(combined_df['Contract'] == 'T7531')]
trial['Item Code'].unique()
# trial.to_excel('Check_T7531.xlsx')

#### Filter Pay items and data after latest cut off

##### Pay Items filter

In [147]:
import pandas as pd
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from datetime import datetime

Master_Pay_Item_List=pd.read_excel(root_dir + r"\\FDOT Master Pay Items.xlsx")
Master_Pay_Item_List.columns = Master_Pay_Item_List.iloc[1]
Master_Pay_Item_List = Master_Pay_Item_List.iloc[2:]
Master_Pay_Item_List = Master_Pay_Item_List[Master_Pay_Item_List['Acme Vlookup']== 'Include']
Master_Pay_Item_List['Item Number'] = Master_Pay_Item_List['Item Number'].str.strip().str.replace(r'\s+', ' ', regex=True)
# Create lowercased description columns for merging
# combined_df['Item Description Lower'] = combined_df['Item Description'].str.lower()
# Master_Pay_Item_List['Item Description Lower'] = Master_Pay_Item_List['Item Description'].str.lower()

filtered_df = combined_df.merge(
	Master_Pay_Item_List[['Item Number']],
	left_on=['Item Code'],
	right_on=['Item Number'],
	how='inner'
).drop(columns=['Item Number'])

##### Filtered Data after latest cut off date

In [148]:
# Read Cut-off Dates
from datetime import datetime
cutoff_data = pd.read_excel(root_dir + r"\\FDOT Monthly CutOff Dates.xlsx")
current_month = (datetime.now().replace(day=1)).strftime('%B')
print("current_month", current_month)
last_month = (datetime.now().replace(day=1) - pd.DateOffset(months=1)).strftime('%B')
print("last_Month", last_month)
last_cutoff_date = cutoff_data[cutoff_data['Month'] == last_month]['Cutoff date'].values[0]
current_cutoff_date = cutoff_data[cutoff_data['Month'] == current_month]['Cutoff date'].values[0]
# final_df = filtered_df[(filtered_df['EstimateEnd Date']>last_cutoff_date) & (filtered_df['EstimateEnd Date']<=current_cutoff_date)]
final_df = filtered_df[(filtered_df['EstimateEnd Date']>last_cutoff_date)]
yesterday = (datetime.today() - pd.DateOffset(days=1)).strftime('%Y-%b-%d')


current_month October
last_Month September


##### Append Final Acceptance Date to Final df

In [ ]:
contracts_df_payment = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Latest.xlsx")
contracts_df_payment.columns
print(contracts_df_payment.shape)   
contracts_df_payment = contracts_df_payment[['Contract ID', 'Final Acceptance Date']].drop_duplicates()      
print(contracts_df_payment.shape)
contracts_df_payment.rename(columns={"Contract ID": "Contract"}, inplace=True)        
contracts_df_payment['Final Acceptance Date'] = pd.to_datetime(contracts_df_payment['Final Acceptance Date']).dt.strftime('%m-%d-%Y')
final_df = final_df.merge(contracts_df_payment, how='left', on = "Contract")           

print(final_df.shape)
final_df = final_df.drop_duplicates()
print(final_df.shape)

(1501, 30)
(1501, 2)


In [151]:
# final_df.to_csv(os.path.join(root_dir, 'FDOT_Output_Data_' + str(yesterday) + '.csv'), index=False)
final_df.to_excel(os.path.join(root_dir, 'FDOT_Output_Data_' + str(yesterday) + '.xlsx'), index=False)

#### Append Daily Fresh Data for final combined output

In [27]:
import os
os.path.join(root_dir, 'FDOT_Output_Data_' + str(datetime.today().strftime('%Y-%b-%d')) + '.csv')

'C:\\Users\\IlaBarshilia\\ACME Barricades\\FDOT Web Scraping - Source Data\\FDOT_Output_Data_2025-Aug-16.csv'

#### Read Active Contracts Data and compare Contracts ID in it with combined output and No Data

In [8]:
import re

# Replace with the path to your HTML file
file_path = r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation\FDOT_WebScraping_2025-Jun-27.html'

# Read the HTML content
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# Regular expression to find patterns like [E1W93] No estimate detail report available. Skipping.
pattern = r"\[([^\[\]]+)\]\s+No estimate detail report available\. Skipping\."

# Find all matching IDs
No_Data_Active_Contracts_List = re.findall(pattern, content)
No_Data_Active_Contracts_List = [item for item in No_Data_Active_Contracts_List if "span class" not in item]

# Read the combined output file
Combined_Downloaded_Active_df=pd.read_excel(r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation\combined_FDOT_output_2025-Jun-27.xlsx',
                  engine='openpyxl')

Combined_Downloaded_Active_df.columns = [col.replace('\n', ' ').lstrip() for col in Combined_Downloaded_Active_df.columns]
Downloaded_Active_Contract_List = Combined_Downloaded_Active_df.Contract.unique().tolist()

All_Active_Contracts_df=pd.read_excel(r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation\FDOT_active_contracts_2025-Jun-27.xlsx')
All_Active_Contracts_List = All_Active_Contracts_df['Contract ID'].unique().tolist()

print("Unique Contract IDs in combined FDOT Output on 2025-Jun-27: ", Combined_Downloaded_Active_df.Contract.nunique())
print("Unique Contract IDs in Active Contracts on 2025-Jun-27: ", All_Active_Contracts_df['Contract ID'].nunique())
print("Unique Contract IDs in Active Contracts with No Data: ", len(set(No_Data_Active_Contracts_List)))

common_elements = list(set(Downloaded_Active_Contract_List) & set(No_Data_Active_Contracts_List))
print("\nContract IDs repeated in Downloaded Data and no data HTML from python output:", common_elements)


if Combined_Downloaded_Active_df.Contract.nunique() + len(set(No_Data_Active_Contracts_List)) == All_Active_Contracts_df['Contract ID'].nunique():
    print("All contracts in the combined output have been downloaded successfully.")
elif Combined_Downloaded_Active_df.Contract.nunique() + len(set(No_Data_Active_Contracts_List)) < All_Active_Contracts_df['Contract ID'].nunique():
    print("There are some contracts in the active contracts list that have not been downloaded successfully.")
    print("Please check the HTML file for details on which contracts were skipped.")
else:
    print("There are some contracts which appear in no data count and combined output both, check all contracts in detail.")


Combined_Active_Data_NoData_List = set(Downloaded_Active_Contract_List + No_Data_Active_Contracts_List)
not_present_in_Downloaded_List = [item for item in All_Active_Contracts_List if item not in Combined_Active_Data_NoData_List]
not_present_in_Active_List = [item for item in Combined_Active_Data_NoData_List if item not in All_Active_Contracts_List]
print('Çheck these contract IDs:', not_present_in_Downloaded_List)

KeyboardInterrupt: 

#### Read ACME DOT Jobs and compare combined data with ACME's

In [1]:
import pandas as pd
combined_df=pd.read_excel(r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation\combined_FDOT_output_2025-Jun-27.xlsx',
                  engine='openpyxl')
combined_df.columns = [col.replace('\n', '').strip() for col in combined_df.columns]



ACME_dot_jobs=pd.read_excel(r'C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Acme DOT Jobs.xlsx',
                            usecols=[0])
ACME_dot_jobs.drop_duplicates(inplace=True)

All_Jobs = ACME_dot_jobs.merge(combined_df, left_on='Job', right_on='Contract', how='outer',indicator=True)
Count_Jobs_not_in_FDOT = All_Jobs[All_Jobs['_merge'] == 'left_only']['Job'].nunique()

Jobs_not_in_Downloaded_Contracts=All_Jobs[All_Jobs['_merge'] == 'left_only']['Job'].unique().tolist()
print("Jobs not in Downloaded Contracts:", len(set(Jobs_not_in_Downloaded_Contracts)))
print("Total Jobs in ACME DOT:", ACME_dot_jobs['Job'].nunique())

All_Contracts_df=pd.read_excel(r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation\FDOT_All_Contracts_2025-Jun-27.xlsx')
Jobs_not_in_FDOT_but_in_All_Contracts = [item for item in Jobs_not_in_Downloaded_Contracts if item in All_Contracts_df['Contract ID'].values]
print(len(set(Jobs_not_in_FDOT_but_in_All_Contracts)))

Jobs_not_in_AllContracts = ACME_dot_jobs[~ACME_dot_jobs['Job'].isin(All_Contracts_df['Contract ID'])]['Job'].tolist()
print(len(set(Jobs_not_in_AllContracts)))
All_Contracts_df['Contract ID'].nunique()

KeyboardInterrupt: 